In [ ]:
import pandas as pd
import numpy as np

real_time_data = pd.read_csv("fetched_gold_metal_economy_headlines.csv")

In [3]:
real_time_data.head()

,Date,Headline
0,2025-04-26,Is UK prime property making a comeback as a sa...
1,2025-04-27,Metals experienced a near across-the-board dec...
2,2025-04-27,Does Wall Street's Sell-Off Have You Intereste...
3,2025-04-27,The safe haven asset investors are flocking to...
4,2025-04-27,"Should You Buy Bitcoin While It's Under $95,000?"


In [ ]:
real_time_data["Date"].unique()

array(['2025-04-26', '2025-04-27', '2025-04-28'], dtype=object)

In [ ]:
real_time_data.columns = ["Dates", "News"]

In [6]:
real_time_data.head()

,Dates,News
0,2025-04-26,Is UK prime property making a comeback as a sa...
1,2025-04-27,Metals experienced a near across-the-board dec...
2,2025-04-27,Does Wall Street's Sell-Off Have You Intereste...
3,2025-04-27,The safe haven asset investors are flocking to...
4,2025-04-27,"Should You Buy Bitcoin While It's Under $95,000?"


## 2. Sentiment

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
from tqdm import tqdm
from torch.nn.functional import softmax
import pandas as pd


def sentiment_analysis(df):
    """
    df -> This is a dataframe that consists of all the NY times headline that we have got
    """
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

    # news_subset_df = df[["Dates", "News"]]

    # news_subset_df["Dates"] = pd.to_datetime(
    #     news_subset_df["Dates"], format="mixed", dayfirst=True, errors="coerce"
    # )

    grouped = df.groupby("Dates")["News"].apply(list)

    daily_sentiments = []

    for date, news_list in tqdm(grouped.items(), desc="Computing FinBERT sentiment"):
        logits_list = []

        for sentence in news_list:
            inputs = tokenizer(
                sentence, return_tensors="pt", truncation=True, padding=True
            )
            inputs = {k: v.to("cpu") for k, v in inputs.items()}
            model.to("cpu")
            with torch.no_grad():
                logits = model(**inputs).logits.squeeze()
            logits_list.append(logits)

        stacked_logits = torch.stack(logits_list)
        avg_logits = torch.mean(stacked_logits, dim=0)
        probabilities = softmax(avg_logits, dim=0)

        labels = model.config.id2label
        final_probs = {labels[i]: float(probabilities[i]) for i in range(len(labels))}
        final_sentiment = max(final_probs, key=final_probs.get)

        daily_sentiments.append(
            {
                "Date": date,
                "Positive": final_probs.get("positive", 0.0),
                "Neutral": final_probs.get("neutral", 0.0),
                "Negative": final_probs.get("negative", 0.0),
                "Final Sentiment": final_sentiment,
            }
        )

    sentiment_group_df = pd.DataFrame(daily_sentiments)

    sentiment_group_df["Date"] = pd.to_datetime(
        sentiment_group_df["Date"], dayfirst=True, errors="coerce"
    )
    sentiment_group_df = sentiment_group_df.sort_values("Date").reset_index(drop=True)

    # Sentiment returned as a pos, neg, nuetral and overall sentiment
    return sentiment_group_df

/Users/visheshgupta/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
real_time_data_sentiment = sentiment_analysis(real_time_data)

Device set to use mps:0
Computing FinBERT sentiment: 3it [00:00,  4.87it/s]
/var/folders/mc/2wjfdchj6vsffbrpfbfgqw4w0000gn/T/ipykernel_21300/1561757246.py:60: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  sentiment_group_df["Date"] = pd.to_datetime(


In [9]:
real_time_data_sentiment

,Date,Positive,Neutral,Negative,Final Sentiment
0,2025-04-26,0.416320,0.563284,0.020397,neutral
1,2025-04-27,0.188708,0.672676,0.138616,neutral
2,2025-04-28,0.397852,0.305925,0.296224,positive


## 3. Clustering

In [ ]:
import pickle

with open("topic_model.pkl", "rb") as file:
    topic_model = pickle.load(file)

topic_group_map = pd.read_csv("topic_group_map.csv")

In [11]:
import re

months = r"\b(january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|jun|jul|aug|sep|oct|nov|dec)\b"
directions = r"\b(up|down|higher|lower|rise|rises|fall|falls|gain|gains|loses|loss|rebound|slip|climb|surge|drop|drops|edged|edges|recover|recovery|recovers|flat)\b"
numbers = r"[\d\.,]+[%$]?|\d{1,3}(,\d{3})*(\.\d+)?|\d+"
symbols = r"\/oz|rs|bn|usd|\$|%|oz"

In [ ]:
def classify_headline(row):
    cleaned = re.sub(months, "", row.lower())
    cleaned = re.sub(directions, "", cleaned)
    cleaned = re.sub(numbers, "", cleaned)
    cleaned = re.sub(symbols, "", cleaned)
    cleaned = re.sub(r"[^\w\s]", "", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # Get topic and probability from the model
    topic, prob = topic_model.transform([cleaned])

    if topic == -1 or len(topic) == 0:
        return "noise"
    else:
        # Ensure topic is valid and exists in the map
        try:
            merged_group = topic_group_map.loc[
                topic_group_map["Original_Topic"] == topic[0], "Macro_Group"
            ].values[0]
            return merged_group
        except IndexError:
            return -1


# Apply classification to each row in the 'News' column
real_time_data["Category"] = real_time_data["News"].apply(classify_headline)

real_time_data

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.69it/s]
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
Batches: 100%|██████████| 1/1 [00:00<00:00, 14.72it/s]


,Dates,News,Category
0,2025-04-26,Is UK prime property making a comeback as a sa...,-1
1,2025-04-27,Metals experienced a near across-the-board dec...,3
2,2025-04-27,Does Wall Street's Sell-Off Have You Intereste...,12
3,2025-04-27,The safe haven asset investors are flocking to...,3
4,2025-04-27,"Should You Buy Bitcoin While It's Under $95,000?",3
5,2025-04-27,Asia-Pacific markets trade mixed after China v...,13
6,2025-04-27,"From mining giants to Big Oil, major players a...",-1
7,2025-04-27,Indian Housewife Is The Smartest Fund Manager ...,-1
8,2025-04-28,"Gold falls on firmer dollar, US-China trade te...",6
9,2025-04-28,Thai bonds see rush of inflows on rate-cut bet...,-1


In [ ]:
category_to_topic = {
    1: "topic_gold_climbs_in_trading",
    2: "topic_market_rates_and_gold_demand",
    3: "topic_global_investment_and_rate_impact",
    4: "topic_gold_settlement_price_correction",
    5: "topic_time_sensitive_prediction",
    6: "topic_economic_data_and_gold_performance",
    7: "topic_gold_trading_momentum_continues",
    8: "topic_early_gold_trading_activity",
    9: "topic_gold_spot_price_outlook",
    10: "topic_comex_gold_closing_prices",
    11: "topic_gold_market_turning_points",
    12: "topic_global_cues_and_gold_stability",
    13: "topic_china_fed_and_gold_focus",
    14: "topic_gold_prices_and_metal_shares",
    15: "topic_gold_near_monthly_lows",
    16: "topic_gold_settlement_price_correction",
    17: "topic_gold_holds_as_data_releases",
    18: "topic_rate_hikes/cuts_price_impact",
    -1: "topic_noise",
}

# Map Category to Topic
real_time_data["Topic"] = real_time_data["Category"].map(category_to_topic)

real_time_data

,Dates,News,Category,Topic
0,2025-04-26,Is UK prime property making a comeback as a sa...,-1,topic_noise
1,2025-04-27,Metals experienced a near across-the-board dec...,3,topic_global_investment_and_rate_impact
2,2025-04-27,Does Wall Street's Sell-Off Have You Intereste...,12,topic_global_cues_and_gold_stability
3,2025-04-27,The safe haven asset investors are flocking to...,3,topic_global_investment_and_rate_impact
4,2025-04-27,"Should You Buy Bitcoin While It's Under $95,000?",3,topic_global_investment_and_rate_impact
5,2025-04-27,Asia-Pacific markets trade mixed after China v...,13,topic_china_fed_and_gold_focus
6,2025-04-27,"From mining giants to Big Oil, major players a...",-1,topic_noise
7,2025-04-27,Indian Housewife Is The Smartest Fund Manager ...,-1,topic_noise
8,2025-04-28,"Gold falls on firmer dollar, US-China trade te...",6,topic_economic_data_and_gold_performance
9,2025-04-28,Thai bonds see rush of inflows on rate-cut bet...,-1,topic_noise


In [14]:
import pandas as pd

# --- Step 1: Parse dates safely ---
real_time_data["Dates"] = pd.to_datetime(
    real_time_data["Dates"], dayfirst=True, errors="coerce"
)

# --- Step 2: Drop rows with invalid dates ---
real_time_data = real_time_data.dropna(subset=["Dates"])

# --- Step 3: Convert to date only (no time component) ---
real_time_data["Dates"] = real_time_data["Dates"].dt.date

# --- Step 4: Group by Dates and Topic_Label, count headline frequency ---
daily_counts = (
    real_time_data.groupby(["Dates", "Topic"]).size().reset_index(name="count")
)

In [15]:
daily_counts

,Dates,Topic,count
0,2025-04-26,topic_noise,1
1,2025-04-27,topic_china_fed_and_gold_focus,1
2,2025-04-27,topic_global_cues_and_gold_stability,1
3,2025-04-27,topic_global_investment_and_rate_impact,3
4,2025-04-27,topic_noise,2
5,2025-04-28,topic_economic_data_and_gold_performance,2
6,2025-04-28,topic_global_investment_and_rate_impact,2
7,2025-04-28,topic_noise,8


In [16]:
all_topics = category_to_topic.values()

In [17]:
len(all_topics)

19

In [18]:
all_topics

dict_values(['topic_gold_climbs_in_trading', 'topic_market_rates_and_gold_demand', 'topic_global_investment_and_rate_impact', 'topic_gold_settlement_price_correction', 'topic_time_sensitive_prediction', 'topic_economic_data_and_gold_performance', 'topic_gold_trading_momentum_continues', 'topic_early_gold_trading_activity', 'topic_gold_spot_price_outlook', 'topic_comex_gold_closing_prices', 'topic_gold_market_turning_points', 'topic_global_cues_and_gold_stability', 'topic_china_fed_and_gold_focus', 'topic_gold_prices_and_metal_shares', 'topic_gold_near_monthly_lows', 'topic_gold_settlement_price_correction', 'topic_gold_holds_as_data_releases', 'topic_rate_hikes/cuts_price_impact', 'topic_noise'])

In [ ]:
# --- Step 1: Aggregate any duplicate date-topic pairs first ---
daily_counts_aggregated = (
    daily_counts.groupby(["Dates", "Topic"])["count"].sum().reset_index()
)

# --- Step 2: Create complete date-topic combinations ---
all_dates = daily_counts_aggregated["Dates"].unique()
complete_combinations = pd.MultiIndex.from_product(
    [all_dates, all_topics], names=["Dates", "Topic"]
).to_frame(index=False)

# --- Step 3: Merge with aggregated counts ---
complete_counts = pd.merge(
    complete_combinations, daily_counts_aggregated, on=["Dates", "Topic"], how="left"
).fillna(0)

# Convert counts to integers
complete_counts["count"] = complete_counts["count"].astype(int)

In [ ]:
complete_counts["Topic"].nunique()

18

In [ ]:
# --- Step 4: Pivot (now no duplicates exist) ---
frequency_embedding = complete_counts.pivot(
    index="Dates", columns="Topic", values="count"
).fillna(
    0
)  # Additional safety

# --- Step 5: Sanitize column names ---
frequency_embedding.columns = [
    str(col)
    .lower()
    .replace(" ", "_")
    .replace("&", "and")
    .replace(",", "")
    .replace("/", "_")
    for col in frequency_embedding.columns
]

# --- Step 6: Reset index ---
frequency_embedding = frequency_embedding.reset_index()

ValueError: Index contains duplicate entries, cannot reshape